# PCA diagnostics — RNA

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.decomposition import PCA

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

# Resolve this analysis folder (paper/01_rna) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '00_build_metadata.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/01_rna')

DIR = _here()
STAGES = ['ESC', 'DE', 'HB', 'iHEP', 'mHEP']
STAGE_COLORS = json.loads((DIR / 'data' / 'stage_colors.json').read_text())


def plot_pca(pcs, ev, stage_of, rep_of, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    for ax, (a, b) in zip(axes, [(0, 1), (0, 2)]):
        for st in STAGES:
            idx = [i for i, s in enumerate(stage_of) if s == st]
            ax.scatter(pcs[idx, a], pcs[idx, b], c=STAGE_COLORS[st], label=st,
                       s=128, edgecolors='none', linewidths=0.5, zorder=3)
        for i in range(len(stage_of)):
            ax.annotate(rep_of[i], (pcs[i, a], pcs[i, b]), fontsize=8,
                        ha='center', va='center', zorder=4)
        ax.set_xlabel(f'PC{a+1} ({ev[a]:.1%})')
        ax.set_ylabel(f'PC{b+1} ({ev[b]:.1%})')
        ax.axhline(0, color='0.85', lw=0.5); ax.axvline(0, color='0.85', lw=0.5)
    axes[0].set_title('PC1 vs PC2'); axes[1].set_title('PC1 vs PC3')
    axes[0].legend(title='stage', fontsize=8)
    fig.suptitle(title, fontsize=12, y=1.0)
    fig.tight_layout()
    return fig

In [ ]:
vst = pl.read_parquet(DIR / 'results' / 'gene_vst.parquet')

samples = [f'{s}_REP{r}' for s in STAGES for r in (1, 2)]

X = vst.select(samples).to_numpy().T          # samples x genes
pca = PCA(n_components=5).fit(X)
pcs = pca.transform(X)
ev = pca.explained_variance_ratio_

stage_of = [s.rsplit('_', 1)[0] for s in samples]
rep_of = [s.rsplit('_', 1)[1].replace('REP', '') for s in samples]
print('RNA PC var:', '  '.join(f'PC{i+1}={ev[i]:.1%}' for i in range(5)))
fig = plot_pca(pcs, ev, stage_of, rep_of, 'RNA-seq PCA — DESeq2 VST (per replicate)');
fig.savefig(DIR / 'figs' / f'rna_pca.png', dpi=200, bbox_inches='tight')
fig.savefig(DIR / 'figs' / f'rna_pca.pdf', bbox_inches='tight')